# LeetCode 1401 — Circle and Rectangle Overlapping bằng Python

Link đề: https://leetcode.com/problems/circle-and-rectangle-overlapping/ · Độ khó: **Medium** · Chủ đề: Math, Geometry

Skeleton LeetCode:

```python
class Solution:
    def checkOverlap(self, radius: int, xCenter: int, yCenter: int, x1: int, y1: int, x2: int, y2: int) -> bool:
        ...
```

## 1. Mục tiêu học

1. Biết quy bài "hai hình có giao nhau không" về **khoảng cách từ tâm tròn tới hình chữ nhật** và tìm **điểm gần nhất** bằng phép **kẹp (clamp)**.
2. Tránh số thực: so **bình phương khoảng cách** với `r²` — chỉ dùng số nguyên, không sai số.
3. Cài đặt và so sánh 4 cách: duyệt điểm lưới (baseline), clamp, khoảng cách theo trục `max(…, 0, …)`, và chia vùng "hình chữ nhật bo góc".
4. Hiểu vì sao baseline lưới điểm nguyên lại **chính xác** (không chỉ gần đúng) khi dữ liệu là số nguyên.
5. Tránh lỗi: chỉ kiểm tra góc, chỉ kiểm tra tâm, dùng `<` thay `≤` (bỏ sót trường hợp tiếp xúc), tính khoảng cách tới cạnh sai.

## 2. Đề bài & ví dụ

Cho hình tròn bán kính `radius`, tâm `(xCenter, yCenter)`, và hình chữ nhật **song song trục toạ độ** với góc dưới trái `(x1, y1)`, góc trên phải `(x2, y2)`. Trả về `True` nếu hai hình **có chung ít nhất một điểm** (tính cả biên — chạm nhau tại một điểm cũng tính), ngược lại `False`.

**Ví dụ 1:** `radius = 1, xCenter = 0, yCenter = 0, x1 = 1, y1 = -1, x2 = 3, y2 = 1` → `True` (chạm nhau tại `(1, 0)`).

**Ví dụ 2:** `radius = 1, xCenter = 1, yCenter = 1, x1 = 1, y1 = -3, x2 = 2, y2 = -1` → `False` (điểm gần nhất là `(1, -1)`, cách tâm 2 > 1).

**Ví dụ 3:** `radius = 1, xCenter = 0, yCenter = 0, x1 = -1, y1 = 0, x2 = 0, y2 = 1` → `True` (tâm tròn chính là góc dưới phải của hình chữ nhật).

**Ràng buộc:** `1 <= radius <= 2000`, `-10^4 <= xCenter, yCenter <= 10^4`, `-10^4 <= x1 < x2 <= 10^4`, `-10^4 <= y1 < y2 <= 10^4`.

In [1]:
import math
import random
import time
from typing import List

# (radius, xCenter, yCenter, x1, y1, x2, y2, đáp án)
examples = [
    (1, 0, 0, 1, -1, 3, 1, True),
    (1, 1, 1, 1, -3, 2, -1, False),
    (1, 0, 0, -1, 0, 0, 1, True),
]
for *args, want in examples:
    print(args, "->", want)

[1, 0, 0, 1, -1, 3, 1] -> True
[1, 1, 1, 1, -3, 2, -1] -> False
[1, 0, 0, -1, 0, 0, 1] -> True


## 3. Kiến thức nền

### 3.1 Khoảng cách Euclid và bình phương khoảng cách

Khoảng cách giữa `(a, b)` và `(c, d)` là `√((a−c)² + (b−d)²)`. Điểm `P` nằm **trong hoặc trên** đường tròn tâm `O` bán kính `r` ⇔ `|OP| ≤ r` ⇔ `|OP|² ≤ r²` (cả hai vế không âm). Dạng bình phương chỉ dùng **phép nhân số nguyên** — chính xác tuyệt đối.

In [2]:
O, P, r = (0, 0), (3, 4), 5
d2 = (P[0] - O[0]) ** 2 + (P[1] - O[1]) ** 2
print("d² =", d2, "| r² =", r * r, "| trong/trên đường tròn?", d2 <= r * r)
print("math.hypot:", math.hypot(3, 4), "| math.dist:", math.dist(O, P))

d² = 25 | r² = 25 | trong/trên đường tròn? True
math.hypot: 5.0 | math.dist: 5.0


### 3.2 Hàm kẹp (clamp)

`clamp(v, lo, hi) = max(lo, min(v, hi))` đưa `v` về đoạn `[lo, hi]`: nếu `v < lo` thì `lo`, nếu `v > hi` thì `hi`, ngược lại giữ `v`. Trên **một trục**, `clamp(v, lo, hi)` là điểm của đoạn `[lo, hi]` **gần `v` nhất**.

In [3]:
def clamp(v, lo, hi):
    return max(lo, min(v, hi))

for v in [-5, 1, 2, 7]:
    print(f"clamp({v}, 1, 3) = {clamp(v, 1, 3)}")

clamp(-5, 1, 3) = 1
clamp(1, 1, 3) = 1
clamp(2, 1, 3) = 2
clamp(7, 1, 3) = 3


### 3.3 Điểm thuộc hình chữ nhật song song trục

`(x, y)` thuộc hình chữ nhật (kể cả biên) ⇔ `x1 ≤ x ≤ x2` **và** `y1 ≤ y ≤ y2`. Python cho viết gọn `x1 <= x <= x2`.

In [4]:
def in_rect(x, y, x1, y1, x2, y2):
    return x1 <= x <= x2 and y1 <= y <= y2

print(in_rect(1, 0, 1, -1, 3, 1), in_rect(0, 0, 1, -1, 3, 1), in_rect(3, 1, 1, -1, 3, 1))

True False True


### 3.4 Vì sao nên tránh số thực khi so sánh

Phép căn và phép chia số thực có sai số làm tròn. Với bài **chạm biên** (khoảng cách đúng bằng bán kính), so `math.sqrt(d2) <= r` thường vẫn đúng với số nhỏ, nhưng thói quen so bình phương giúp loại bỏ hẳn rủi ro — đặc biệt khi toạ độ lớn.

In [5]:
print(0.1 + 0.2 <= 0.3)                      # sai số kinh điển
big = 10**8 + 1
print(math.sqrt(big * big) == big, math.sqrt(big * big + 1) == big)   # căn của số khác nhau ra cùng giá trị
print("isqrt:", math.isqrt(big * big + 1), "| so bình phương:", big * big + 1 <= big * big)

False
True True
isqrt: 100000001 | so bình phương: False


### 3.5 `math.isqrt` — căn bậc hai nguyên

`math.isqrt(n)` = `⌊√n⌋` tính chính xác bằng số nguyên. Hữu ích khi cần "cột này của hình tròn cao bao nhiêu điểm nguyên" (Bài 5 mục 17).

In [6]:
for n in [0, 1, 15, 16, 17, 10**12 + 5]:
    print(n, "->", math.isqrt(n))

0 -> 0
1 -> 1
15 -> 3
16 -> 4
17 -> 4
1000000000005 -> 1000000


## 4. Ý tưởng nền tảng / trực giác

Hình tròn (kể cả bên trong) = tập các điểm cách tâm `O` không quá `r`. Hai hình có điểm chung ⇔ **có điểm của hình chữ nhật cách `O` không quá `r`** ⇔ **điểm của hình chữ nhật gần `O` nhất** cách `O` không quá `r`.

Vậy chỉ cần tìm điểm gần nhất `Q`. Vì hình chữ nhật song song trục, `Q` tìm **độc lập theo từng trục**: `Q = (clamp(xc, x1, x2), clamp(yc, y1, y2))`.

Tính tay ba ví dụ:

| Ví dụ | Tâm | Q | d² | r² | Kết luận |
|---|---|---|---|---|---|
| 1 | (0, 0) | (clamp(0,1,3), clamp(0,−1,1)) = (1, 0) | 1 | 1 | 1 ≤ 1 ✔ chạm |
| 2 | (1, 1) | (clamp(1,1,2), clamp(1,−3,−1)) = (1, −1) | 4 | 1 | 4 > 1 ✘ |
| 3 | (0, 0) | (clamp(0,−1,0), clamp(0,0,1)) = (0, 0) | 0 | 1 | tâm nằm trên hình chữ nhật ✔ |

In [7]:
for r, xc, yc, x1, y1, x2, y2, want in examples:
    qx, qy = clamp(xc, x1, x2), clamp(yc, y1, y2)
    d2 = (qx - xc) ** 2 + (qy - yc) ** 2
    print(f"tâm=({xc},{yc}) Q=({qx},{qy}) d²={d2} r²={r * r} -> {d2 <= r * r} (đề: {want})")

tâm=(0,0) Q=(1,0) d²=1 r²=1 -> True (đề: True)
tâm=(1,1) Q=(1,-1) d²=4 r²=1 -> False (đề: False)
tâm=(0,0) Q=(0,0) d²=0 r²=1 -> True (đề: True)


## 5. Cách 1 — Brute Force / baseline: duyệt điểm lưới nguyên

### Ý tưởng
Duyệt mọi **điểm có toạ độ nguyên** nằm trong hình chữ nhật **và** trong khung vuông bao quanh hình tròn `[xc−r, xc+r] × [yc−r, yc+r]`; nếu có điểm nào thoả `(x−xc)² + (y−yc)² ≤ r²` thì trả `True`.

**Vì sao chỉ xét điểm nguyên mà vẫn chính xác?** Mọi dữ liệu đầu vào là số nguyên ⇒ điểm gần nhất `Q = (clamp(xc,x1,x2), clamp(yc,y1,y2))` cũng có toạ độ **nguyên**. Nếu hai hình giao nhau thì `Q` nằm trong hình tròn ⇒ có ít nhất một điểm nguyên chung. (Nếu dữ liệu là số thực, cách này chỉ còn là **xấp xỉ**.)

In [8]:
def overlap_grid(r, xc, yc, x1, y1, x2, y2):
    lx, hx = max(x1, xc - r), min(x2, xc + r)      # giao theo trục x
    ly, hy = max(y1, yc - r), min(y2, yc + r)      # giao theo trục y
    for x in range(lx, hx + 1):
        for y in range(ly, hy + 1):
            if (x - xc) ** 2 + (y - yc) ** 2 <= r * r:
                return True
    return False

for *args, want in examples:
    print(overlap_grid(*args), "| mong đợi", want)

True | mong đợi True
False | mong đợi False
True | mong đợi True


### Độ phức tạp và nút thắt

- Time `O(min(W, 2r) · min(H, 2r))` với `W = x2 − x1`, `H = y2 − y1` — cận trên thô `(2r+1)² ≈ 1,6·10⁷` điểm.
- Khi **không** giao, ta phải duyệt hết phần giao giữa hình chữ nhật và khung vuông; phần đó nằm ở "góc" ngoài hình tròn, cạnh cỡ `0,29·r` ⇒ với `r = 2000` là ~3,4·10⁵ điểm cho **một** truy vấn.
- Space `O(1)`.

Đo trường hợp đó (`r = 2000`, góc hình chữ nhật `(1415, 1415)` cách tâm `≈ 2001,1 > 2000`):

In [9]:
case = (2000, 0, 0, 1415, 1415, 10000, 10000)   # 1415² · 2 = 4004450 > 2000² = 4000000
t = time.perf_counter()
print(overlap_grid(*case), f"{time.perf_counter() - t:.3f}s")

False 0.034s


Nút thắt: ta thử **hàng nghìn điểm** trong khi chỉ một điểm duy nhất — điểm gần nhất — quyết định đáp án.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight — tách theo trục.** Bình phương khoảng cách `(x − xc)² + (y − yc)²` là **tổng hai phần độc lập**: một phần chỉ phụ thuộc `x`, một phần chỉ phụ thuộc `y`. Trên hình chữ nhật, `x` chạy tự do trong `[x1, x2]` và `y` chạy tự do trong `[y1, y2]` ⇒ cực tiểu tổng = cực tiểu từng phần ⇒ `x = clamp(xc, x1, x2)`, `y = clamp(yc, y1, y2)`.

**Viết lại theo khoảng cách trục:**
- `dx = max(x1 − xc, 0, xc − x2)`: nếu tâm ở bên trái hình chữ nhật thì `x1 − xc > 0`; bên phải thì `xc − x2 > 0`; ở giữa thì `0`.
- Tương tự `dy`.
- Giao nhau ⇔ `dx² + dy² ≤ r²`.

**Quy tắc "nếu… thì…":**
- Nếu tâm nằm trong hình chữ nhật ⇒ `dx = dy = 0` ⇒ luôn giao.
- Nếu tâm nằm trong "dải ngang" `y1 ≤ yc ≤ y2` ⇒ chỉ cần `dx ≤ r`.
- Nếu tâm nằm ở "góc chéo" ⇒ so khoảng cách tới **góc** gần nhất.

## 7. Các cách tiếp theo

### Cách 2 — Điểm gần nhất bằng clamp

**Từng đoạn:**
1. `qx = max(x1, min(xCenter, x2))`, `qy = max(y1, min(yCenter, y2))`.
2. `d2 = (qx − xCenter)² + (qy − yCenter)²`.
3. Trả `d2 <= radius²` — chỉ số nguyên.

In [10]:
def overlap_clamp(r, xc, yc, x1, y1, x2, y2):
    qx = max(x1, min(xc, x2))
    qy = max(y1, min(yc, y2))
    return (qx - xc) ** 2 + (qy - yc) ** 2 <= r * r

for *args, want in examples:
    print(overlap_clamp(*args), "| mong đợi", want)

True | mong đợi True
False | mong đợi False
True | mong đợi True


Time `O(1)`, Space `O(1)`. Bản nộp LeetCode của cách này ở mục 9.

### Cách 3 — Khoảng cách theo trục `max(…, 0, …)`

Cùng bản chất, nhưng viết thẳng khoảng cách trên từng trục mà không cần toạ độ `Q`. Cách viết này hay gặp trong lập trình game (kiểm tra va chạm AABB–circle).

In [11]:
def overlap_axis(r, xc, yc, x1, y1, x2, y2):
    dx = max(x1 - xc, 0, xc - x2)
    dy = max(y1 - yc, 0, yc - y2)
    return dx * dx + dy * dy <= r * r

for *args, want in examples:
    print(overlap_axis(*args), "| mong đợi", want)

True | mong đợi True
False | mong đợi False
True | mong đợi True


In [12]:
class SolutionAxis:
    def checkOverlap(self, radius: int, xCenter: int, yCenter: int,
                     x1: int, y1: int, x2: int, y2: int) -> bool:
        dx = max(x1 - xCenter, 0, xCenter - x2)
        dy = max(y1 - yCenter, 0, yCenter - y2)
        return dx * dx + dy * dy <= radius * radius

print([SolutionAxis().checkOverlap(*a[:-1]) for a in examples])

[True, False, True]


Time `O(1)`, Space `O(1)`.

### Cách 4 — Chia vùng: "hình chữ nhật bo góc" (tổng Minkowski)

Đổi góc nhìn: thay vì phình hình tròn, ta **phình hình chữ nhật** thêm `r` về mọi phía — được hình chữ nhật **bo tròn góc** với bán kính `r`. Hai hình giao nhau ⇔ **tâm tròn nằm trong hình bo góc** đó. Hình bo góc = hợp của:

1. Hình chữ nhật mở rộng **ngang**: `[x1 − r, x2 + r] × [y1, y2]`;
2. Hình chữ nhật mở rộng **dọc**: `[x1, x2] × [y1 − r, y2 + r]`;
3. Bốn hình tròn bán kính `r` tâm tại **bốn góc**.

Cách này dài hơn nhưng rất trực quan, và là cách tư duy tổng quát cho bài "va chạm giữa hai hình" (tổng Minkowski).

In [13]:
def overlap_regions(r, xc, yc, x1, y1, x2, y2):
    if x1 - r <= xc <= x2 + r and y1 <= yc <= y2:          # dải ngang
        return True
    if x1 <= xc <= x2 and y1 - r <= yc <= y2 + r:          # dải dọc
        return True
    for cx, cy in ((x1, y1), (x1, y2), (x2, y1), (x2, y2)):  # bốn góc bo tròn
        if (xc - cx) ** 2 + (yc - cy) ** 2 <= r * r:
            return True
    return False

for *args, want in examples:
    print(overlap_regions(*args), "| mong đợi", want)

True | mong đợi True
False | mong đợi False
True | mong đợi True


Time `O(1)` (tối đa 6 phép kiểm tra), Space `O(1)`.

Vẽ thử bằng ký tự: với hình chữ nhật `[0,4]×[0,2]` và `r = 2`, đánh dấu những tâm `(xc, yc)` nguyên làm hai hình giao nhau (`#` = trong hình chữ nhật gốc, `o` = trong phần bo góc).

In [14]:
def draw_rounded(x1, y1, x2, y2, r):
    for yc in range(y2 + r + 1, y1 - r - 2, -1):
        row = ""
        for xc in range(x1 - r - 1, x2 + r + 2):
            if in_rect(xc, yc, x1, y1, x2, y2):
                row += "#"
            elif overlap_regions(r, xc, yc, x1, y1, x2, y2):
                row += "o"
            else:
                row += "."
        print(row)

draw_rounded(0, 0, 4, 2, 2)

...........
...ooooo...
..ooooooo..
.oo#####oo.
.oo#####oo.
.oo#####oo.
..ooooooo..
...ooooo...
...........


Các góc bị "gọt" tròn: ví dụ tâm `(−2, −2)` cách góc `(0, 0)` là `√8 > 2` ⇒ không giao, nên ô đó là `.`.

## 8. TRACE từng bước

Hàm `trace_overlap` in từng bước của Cách 2 và Cách 3, kèm **vị trí tương đối** của tâm so với hình chữ nhật (một trong 9 vùng: trái/giữa/phải × dưới/giữa/trên).

In [15]:
def trace_overlap(r, xc, yc, x1, y1, x2, y2):
    print(f"Hình tròn: tâm ({xc},{yc}), r={r} | Hình chữ nhật: ({x1},{y1})-({x2},{y2})")
    hx = "trái" if xc < x1 else ("phải" if xc > x2 else "giữa")
    vy = "dưới" if yc < y1 else ("trên" if yc > y2 else "giữa")
    print(f"  vùng của tâm: ngang={hx}, dọc={vy}")
    qx = max(x1, min(xc, x2)); qy = max(y1, min(yc, y2))
    print(f"  clamp: qx = max({x1}, min({xc}, {x2})) = {qx}; qy = max({y1}, min({yc}, {y2})) = {qy}")
    dx = max(x1 - xc, 0, xc - x2); dy = max(y1 - yc, 0, yc - y2)
    print(f"  khoảng cách trục: dx = max({x1 - xc}, 0, {xc - x2}) = {dx}; dy = max({y1 - yc}, 0, {yc - y2}) = {dy}")
    d2 = dx * dx + dy * dy
    print(f"  d² = {dx}² + {dy}² = {d2} {'<=' if d2 <= r * r else '>'} r² = {r * r} -> {d2 <= r * r}")
    return d2 <= r * r

for *args, want in examples:
    trace_overlap(*args)
    print()

Hình tròn: tâm (0,0), r=1 | Hình chữ nhật: (1,-1)-(3,1)
  vùng của tâm: ngang=trái, dọc=giữa
  clamp: qx = max(1, min(0, 3)) = 1; qy = max(-1, min(0, 1)) = 0
  khoảng cách trục: dx = max(1, 0, -3) = 1; dy = max(-1, 0, -1) = 0
  d² = 1² + 0² = 1 <= r² = 1 -> True

Hình tròn: tâm (1,1), r=1 | Hình chữ nhật: (1,-3)-(2,-1)
  vùng của tâm: ngang=giữa, dọc=trên
  clamp: qx = max(1, min(1, 2)) = 1; qy = max(-3, min(1, -1)) = -1
  khoảng cách trục: dx = max(0, 0, -1) = 0; dy = max(-4, 0, 2) = 2
  d² = 0² + 2² = 4 > r² = 1 -> False

Hình tròn: tâm (0,0), r=1 | Hình chữ nhật: (-1,0)-(0,1)
  vùng của tâm: ngang=giữa, dọc=giữa
  clamp: qx = max(-1, min(0, 0)) = 0; qy = max(0, min(0, 1)) = 0
  khoảng cách trục: dx = max(-1, 0, 0) = 0; dy = max(0, 0, -1) = 0
  d² = 0² + 0² = 0 <= r² = 1 -> True



Trace thêm hai tình huống đặc biệt: **hình tròn nằm trọn trong hình chữ nhật** (không góc nào trong tròn) và **hình chữ nhật nằm trọn trong hình tròn**.

In [16]:
trace_overlap(1, 0, 0, -5, -5, 5, 5)
print()
trace_overlap(10, 0, 0, -1, -1, 1, 1)
print()
trace_overlap(3, 5, 5, 0, 0, 2, 2)        # vùng góc chéo: d² = 9 + 9 = 18 > 9

Hình tròn: tâm (0,0), r=1 | Hình chữ nhật: (-5,-5)-(5,5)
  vùng của tâm: ngang=giữa, dọc=giữa
  clamp: qx = max(-5, min(0, 5)) = 0; qy = max(-5, min(0, 5)) = 0
  khoảng cách trục: dx = max(-5, 0, -5) = 0; dy = max(-5, 0, -5) = 0
  d² = 0² + 0² = 0 <= r² = 1 -> True

Hình tròn: tâm (0,0), r=10 | Hình chữ nhật: (-1,-1)-(1,1)
  vùng của tâm: ngang=giữa, dọc=giữa
  clamp: qx = max(-1, min(0, 1)) = 0; qy = max(-1, min(0, 1)) = 0
  khoảng cách trục: dx = max(-1, 0, -1) = 0; dy = max(-1, 0, -1) = 0
  d² = 0² + 0² = 0 <= r² = 100 -> True

Hình tròn: tâm (5,5), r=3 | Hình chữ nhật: (0,0)-(2,2)
  vùng của tâm: ngang=phải, dọc=trên
  clamp: qx = max(0, min(5, 2)) = 2; qy = max(0, min(5, 2)) = 2
  khoảng cách trục: dx = max(-5, 0, 3) = 3; dy = max(-5, 0, 3) = 3
  d² = 3² + 3² = 18 > r² = 9 -> False


False

| Tình huống | Vùng tâm | dx | dy | d² | r² | Kết quả |
|---|---|---|---|---|---|---|
| tròn trong chữ nhật | giữa/giữa | 0 | 0 | 0 | 1 | True |
| chữ nhật trong tròn | giữa/giữa | 0 | 0 | 0 | 100 | True |
| góc chéo, xa | phải/trên | 3 | 3 | 18 | 9 | False |

## 9. Code nộp LeetCode

In [17]:
class Solution:
    def checkOverlap(self, radius: int, xCenter: int, yCenter: int,
                     x1: int, y1: int, x2: int, y2: int) -> bool:
        # điểm của hình chữ nhật gần tâm nhất (kẹp từng trục)
        qx = max(x1, min(xCenter, x2))
        qy = max(y1, min(yCenter, y2))
        # so bình phương khoảng cách -> chỉ dùng số nguyên, tính cả trường hợp chạm biên
        return (qx - xCenter) ** 2 + (qy - yCenter) ** 2 <= radius * radius

for *args, want in examples:
    assert Solution().checkOverlap(*args) == want
print("Solution đúng với 3 ví dụ của đề")

Solution đúng với 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

**Bước 1 — quy về điểm gần nhất.** Hình tròn đóng `D = {P : |OP| ≤ r}`. Hình chữ nhật đóng `R`. `D ∩ R ≠ ∅` ⇔ `∃P ∈ R: |OP| ≤ r` ⇔ `min_{P∈R} |OP| ≤ r`. Cực tiểu tồn tại vì `R` đóng và bị chặn.

**Bước 2 — điểm gần nhất là clamp.** Với `P = (x, y) ∈ R`: `|OP|² = (x − xc)² + (y − yc)²`. Hai số hạng phụ thuộc **riêng** `x` và **riêng** `y`, và ràng buộc của `R` là `x ∈ [x1, x2]`, `y ∈ [y1, y2]` **độc lập** ⇒ cực tiểu tổng = cực tiểu từng số hạng. Hàm `(x − xc)²` trên đoạn `[x1, x2]` đạt cực tiểu tại điểm của đoạn gần `xc` nhất = `clamp(xc, x1, x2)` (hàm bậc hai lồi, đối xứng quanh `xc`).

**Bước 3 — so bình phương.** `|OQ| ≤ r ⇔ |OQ|² ≤ r²` vì cả hai không âm. Toàn số nguyên ⇒ không sai số.

**Cách 3** chỉ là viết lại: `|qx − xc| = max(x1 − xc, 0, xc − x2)` (xét ba trường hợp `xc < x1`, `x1 ≤ xc ≤ x2`, `xc > x2`).

**Cách 4** (tổng Minkowski): `D ∩ R ≠ ∅` ⇔ `O ∈ R ⊕ B(0, r)` (hình chữ nhật phình thêm `r`). Tập này đúng bằng hợp hai dải và bốn hình tròn góc: điểm `O` có `dx = 0` hoặc `dy = 0` thì điều kiện thành `dy ≤ r` hoặc `dx ≤ r` (hai dải); nếu cả hai khác 0 thì `Q` là một góc (bốn hình tròn).

**Cách 1** (lưới) chính xác vì `Q` có toạ độ nguyên khi dữ liệu nguyên, và `Q` luôn nằm trong vùng duyệt nếu `|OQ| ≤ r`.

## 11. Phân tích Time / Space

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Lưới điểm nguyên | `O(min(W,2r)·min(H,2r))` ≤ ~1,6·10⁷ | `O(1)` | chỉ chính xác khi dữ liệu nguyên |
| 2. Clamp | `O(1)` | `O(1)` | mặc định |
| 3. Khoảng cách trục | `O(1)` | `O(1)` | cùng bản chất với 2 |
| 4. Chia vùng bo góc | `O(1)` (≤ 6 kiểm tra) | `O(1)` | trực quan, tổng quát hoá Minkowski |

Với ràng buộc, `|qx − xc| ≤ 2·10⁴` ⇒ `d² ≤ 8·10⁸` — Python không lo tràn số; ở C++/Java vẫn vừa `int` 32-bit (≤ 2,1·10⁹), nhưng nên dùng `long` cho an toàn.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
ALL = {
    "grid": overlap_grid,
    "clamp": overlap_clamp,
    "axis": overlap_axis,
    "axis_submit": lambda *a: SolutionAxis().checkOverlap(*a),
    "regions": overlap_regions,
    "submit": lambda *a: Solution().checkOverlap(*a),
}
edge = [
    (1, 0, 0, -5, -5, 5, 5, True),       # tròn nằm trong chữ nhật
    (10, 0, 0, -1, -1, 1, 1, True),      # chữ nhật nằm trong tròn
    (2, 0, 0, 1, -5, 3, 5, True),        # cắt qua giữa cạnh trái, không góc nào trong tròn
    (5, 0, 0, 3, 4, 10, 10, True),       # chạm đúng góc (3,4): 9 + 16 = 25
    (4, 0, 0, 3, 4, 10, 10, False),      # góc (3,4) cách 5 > 4
    (1, 0, 0, 2, -1, 3, 1, False),       # cách cạnh trái 2 > 1
    (2, 0, 0, 2, 2, 3, 3, False),        # góc chéo: 8 > 4
    (3, 0, 0, 2, 2, 3, 3, True),         # góc chéo: 8 <= 9
    (2000, 10000, 10000, -10000, -10000, -9999, -9999, False),   # cực xa
    (2000, 0, 0, 2000, -1, 2001, 1, True),                       # chạm tại (2000, 0)
]
for *a, want in edge:
    assert overlap_grid(*a) == want, a
for name, f in ALL.items():
    for *a, want in examples + edge:
        assert f(*a) == want, (name, a)
print("Mọi cách đúng với", len(examples + edge), "test (nhãn edge kiểm chứng bằng baseline lưới)")

Mọi cách đúng với 13 test (nhãn edge kiểm chứng bằng baseline lưới)


### 12.2 Đối chiếu ngẫu nhiên với baseline lưới (seed cố định)

Toạ độ nhỏ để baseline chạy nhanh và để **thường xuyên rơi vào trường hợp chạm biên**.

In [19]:
rng = random.Random(1401)
cnt = {True: 0, False: 0}
for _ in range(5000):
    r = rng.randint(1, 8)
    xc, yc = rng.randint(-12, 12), rng.randint(-12, 12)
    x1 = rng.randint(-12, 11); x2 = rng.randint(x1 + 1, 12)
    y1 = rng.randint(-12, 11); y2 = rng.randint(y1 + 1, 12)
    a = (r, xc, yc, x1, y1, x2, y2)
    want = overlap_grid(*a)
    cnt[want] += 1
    for name, f in ALL.items():
        assert f(*a) == want, (name, a)
print("5000 bộ ngẫu nhiên khớp baseline; phân bố đáp án:", cnt)

5000 bộ ngẫu nhiên khớp baseline; phân bố đáp án: {True: 1589, False: 3411}


### 12.3 Kiểm tra riêng các trường hợp **chạm biên** (d² = r² đúng bằng)

Dùng các bộ ba Pythagore để dựng điểm gần nhất cách tâm **đúng bằng** `r`: phải trả `True`; dịch ra thêm 1 đơn vị: phải trả `False`.

In [20]:
triples = [(3, 4, 5), (5, 12, 13), (8, 15, 17), (7, 24, 25), (20, 21, 29), (0, 7, 7)]
for a, b, c in triples:
    for sx in (1, -1):
        for sy in (1, -1):
            # góc gần nhất của hình chữ nhật là (sx*a, sy*b); hình chữ nhật nằm "ngoài" về phía đó
            gx, gy = sx * a, sy * b
            X1, X2 = (gx, gx + 5) if sx > 0 else (gx - 5, gx)
            Y1, Y2 = (gy, gy + 5) if sy > 0 else (gy - 5, gy)
            for name, f in ALL.items():
                assert f(c, 0, 0, X1, Y1, X2, Y2) is True, (name, a, b, c)
                assert f(c - 1, 0, 0, X1, Y1, X2, Y2) is False, (name, a, b, c)
print("Mọi cách xử lý đúng trường hợp chạm biên và hụt 1 đơn vị")

Mọi cách xử lý đúng trường hợp chạm biên và hụt 1 đơn vị


### 12.4 Giới hạn ràng buộc (chỉ các cách `O(1)`)

In [21]:
rng = random.Random(7)
t = time.perf_counter()
for _ in range(100000):
    r = rng.randint(1, 2000)
    xc, yc = rng.randint(-10**4, 10**4), rng.randint(-10**4, 10**4)
    x1 = rng.randint(-10**4, 10**4 - 1); x2 = rng.randint(x1 + 1, 10**4)
    y1 = rng.randint(-10**4, 10**4 - 1); y2 = rng.randint(y1 + 1, 10**4)
    a = (r, xc, yc, x1, y1, x2, y2)
    s = Solution().checkOverlap(*a)
    assert s == overlap_axis(*a) == overlap_regions(*a)
print(f"100000 bộ ở biên ràng buộc: các cách O(1) thống nhất ({time.perf_counter() - t:.2f}s)")

100000 bộ ở biên ràng buộc: các cách O(1) thống nhất (0.39s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra **bốn góc** có nằm trong hình tròn không

Bỏ sót khi hình tròn nằm trọn trong hình chữ nhật, hoặc cắt qua **giữa một cạnh**.

In [22]:
def wrong_corners_only(r, xc, yc, x1, y1, x2, y2):
    return any((cx - xc) ** 2 + (cy - yc) ** 2 <= r * r
               for cx, cy in ((x1, y1), (x1, y2), (x2, y1), (x2, y2)))   # SAI

for a in [(1, 0, 0, -5, -5, 5, 5), (2, 0, 0, 1, -5, 3, 5), (1, 0, 0, 1, -1, 3, 1)]:
    print(a, "| sai:", wrong_corners_only(*a), "| đúng:", Solution().checkOverlap(*a))

(1, 0, 0, -5, -5, 5, 5) | sai: False | đúng: True
(2, 0, 0, 1, -5, 3, 5) | sai: False | đúng: True
(1, 0, 0, 1, -1, 3, 1) | sai: False | đúng: True


Ngay **Ví dụ 1** của đề (chạm tại `(1, 0)` giữa cạnh trái) cũng sai. **Sửa:** dùng điểm gần nhất (clamp).

### Lỗi 2 — "Góc trong tròn **hoặc** tâm trong chữ nhật" vẫn chưa đủ

Thêm điều kiện tâm nằm trong hình chữ nhật sửa được trường hợp "tròn trong chữ nhật", nhưng vẫn sót trường hợp **cắt qua giữa cạnh**.

In [23]:
def wrong_corners_or_center(r, xc, yc, x1, y1, x2, y2):
    if x1 <= xc <= x2 and y1 <= yc <= y2:
        return True
    return wrong_corners_only(r, xc, yc, x1, y1, x2, y2)                # SAI: thiếu cạnh

for a in [(1, 0, 0, -5, -5, 5, 5), (2, 0, 0, 1, -5, 3, 5), (1, 0, 0, 1, -1, 3, 1)]:
    print(a, "| sai:", wrong_corners_or_center(*a), "| đúng:", Solution().checkOverlap(*a))

(1, 0, 0, -5, -5, 5, 5) | sai: True | đúng: True
(2, 0, 0, 1, -5, 3, 5) | sai: False | đúng: True
(1, 0, 0, 1, -1, 3, 1) | sai: False | đúng: True


`(2, 0, 0, 1, −5, 3, 5)`: hình tròn tâm gốc bán kính 2 cắt cạnh trái `x = 1` tại các điểm `(1, ±√3)`, nhưng bốn góc ở rất xa và tâm không nằm trong hình chữ nhật. **Sửa:** clamp xử lý gọn **cả ba** trường hợp.

### Lỗi 3 — Dùng `<` thay vì `<=` (bỏ sót tiếp xúc)

In [24]:
def wrong_strict(r, xc, yc, x1, y1, x2, y2):
    qx = max(x1, min(xc, x2)); qy = max(y1, min(yc, y2))
    return (qx - xc) ** 2 + (qy - yc) ** 2 < r * r                      # SAI: phải là <=

for *a, want in examples + [(5, 0, 0, 3, 4, 10, 10, True)]:
    print(a, "| sai:", wrong_strict(*a), "| đúng:", Solution().checkOverlap(*a))

[1, 0, 0, 1, -1, 3, 1] | sai: False | đúng: True
[1, 1, 1, 1, -3, 2, -1] | sai: False | đúng: False
[1, 0, 0, -1, 0, 0, 1] | sai: True | đúng: True
[5, 0, 0, 3, 4, 10, 10] | sai: False | đúng: True


Đề nói rõ "chạm tại một điểm cũng tính". **Sửa:** `<=`.

### Lỗi 4 — Khoảng cách trục tính bằng "cạnh gần hơn" thay vì clamp

Viết `dx = min(abs(xc − x1), abs(xc − x2))` — quên rằng nếu tâm nằm **giữa** `x1` và `x2` thì khoảng cách theo trục x phải là **0**.

In [25]:
def wrong_nearest_edge(r, xc, yc, x1, y1, x2, y2):
    dx = min(abs(xc - x1), abs(xc - x2))                                # SAI khi x1 <= xc <= x2
    dy = min(abs(yc - y1), abs(yc - y2))
    return dx * dx + dy * dy <= r * r

for a in [(2, 0, 0, -5, 1, 5, 8), (1, 0, 0, -5, -5, 5, 5), (1, 0, 0, 1, -1, 3, 1)]:
    print(a, "| sai:", wrong_nearest_edge(*a), "| đúng:", Solution().checkOverlap(*a))

(2, 0, 0, -5, 1, 5, 8) | sai: False | đúng: True
(1, 0, 0, -5, -5, 5, 5) | sai: False | đúng: True
(1, 0, 0, 1, -1, 3, 1) | sai: False | đúng: True


`(2, 0, 0, −5, 1, 5, 8)`: tâm nằm ngay dưới cạnh đáy `y = 1`, khoảng cách thật = 1 ≤ 2. Bản sai lấy `dx = 5` ⇒ `25 + 1 > 4` ⇒ `False`. **Sửa:** `dx = max(x1 − xc, 0, xc − x2)`.

### Lỗi 5 — Đảo thứ tự `min`/`max` trong clamp

`min(x1, max(xc, x2))` **không** phải clamp: kết quả luôn ≤ `x1`.

In [26]:
def wrong_clamp_order(r, xc, yc, x1, y1, x2, y2):
    qx = min(x1, max(xc, x2))                                           # SAI
    qy = min(y1, max(yc, y2))
    return (qx - xc) ** 2 + (qy - yc) ** 2 <= r * r

print("clamp đúng của 5 vào [1,3]:", max(1, min(5, 3)), "| bản sai:", min(1, max(5, 3)))
for *a, want in examples:
    print(a, "| sai:", wrong_clamp_order(*a), "| đúng:", want)

clamp đúng của 5 vào [1,3]: 3 | bản sai: 1
[1, 0, 0, 1, -1, 3, 1] | sai: False | đúng: True
[1, 1, 1, 1, -3, 2, -1] | sai: False | đúng: False
[1, 0, 0, -1, 0, 0, 1] | sai: True | đúng: True


**Sửa:** nhớ câu "**max của cận dưới, min của cận trên**": `max(lo, min(v, hi))`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Hình tròn và hình X có giao không?"** ⇒ tìm **điểm của X gần tâm nhất**, so với `r`.
- Hình chữ nhật/hộp **song song trục** ⇒ điểm gần nhất = **clamp từng trục**.
- Mọi phép so khoảng cách ⇒ so **bình phương**, tránh `sqrt`.
- Biên có tính hay không? Đọc kỹ đề để chọn `<=` hay `<`.
- Muốn tự kiểm tra bài hình học có dữ liệu nguyên ⇒ viết baseline duyệt **điểm lưới** (chính xác nếu điểm tới hạn có toạ độ nguyên).
- Cách nghĩ "phình một hình, co hình kia thành điểm" (tổng Minkowski) giải được nhiều bài va chạm.

## 15. Biến thể của bài

1. **Hình tròn chứa trọn hình chữ nhật** ⇒ so khoảng cách tới **góc xa nhất** với `r` (Bài 4 mục 17).
2. **Hình chữ nhật chứa trọn hình tròn** ⇒ `x1 ≤ xc − r`, `xc + r ≤ x2`, tương tự trục y.
3. **Giao có diện tích dương** (chạm không tính) ⇒ `d² < r²`.
4. **Đếm điểm nguyên nằm trong cả hai hình** ⇒ duyệt từng cột `x`, dùng `isqrt` (Bài 5).
5. **Hai hình chữ nhật giao nhau** (LeetCode 836 *Rectangle Overlap*) ⇒ giao hai đoạn trên mỗi trục.
6. **Hình chữ nhật xoay** ⇒ đưa tâm tròn về hệ trục của hình chữ nhật (phép quay) rồi clamp.
7. **Hình cầu – hộp 3D** ⇒ clamp thêm trục `z`, công thức y hệt.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Lưới điểm nguyên | thử mọi điểm nguyên chung | `O(r²)` | `O(1)` | baseline kiểm thử (dữ liệu nguyên) |
| 2. Clamp | điểm gần nhất `(clamp x, clamp y)` | `O(1)` | `O(1)` | **mặc định** |
| 3. Khoảng cách trục | `dx = max(x1−xc, 0, xc−x2)` | `O(1)` | `O(1)` | va chạm AABB trong game, mở rộng nhiều chiều |
| 4. Bo góc (Minkowski) | tâm thuộc hình chữ nhật phình `r` | `O(1)` | `O(1)` | giảng giải trực quan; bài va chạm tổng quát |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().checkOverlap(1, 1, 1, 1, -3, 2, -1)` (Ví dụ 2) đổi thành bán kính `2` thì trả về gì? Còn bán kính `2` nhưng `y2 = -2`?

### Bài 2 (tính tay)
Tâm `(7, −3)`, `r = 5`, hình chữ nhật `(0, 0)–(4, 6)`. Tính `Q`, `dx`, `dy`, `d²` và kết luận.

### Bài 3 (điền TODO)
Hoàn thiện `closest_point(xc, yc, x1, y1, x2, y2)` trả về điểm của hình chữ nhật gần tâm nhất.

### Bài 4 (tự viết lại)
Viết `circle_contains_rect(r, xc, yc, x1, y1, x2, y2)`: `True` nếu hình chữ nhật nằm **trọn** trong hình tròn (tính cả biên).

### Bài 5 (mini challenge)
Viết `count_common_lattice(r, xc, yc, x1, y1, x2, y2)`: đếm số điểm **nguyên** nằm trong cả hai hình, chạy `O(min(W, r))` bằng `math.isqrt`.

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def closest_point(xc, yc, x1, y1, x2, y2):
    # TODO: kẹp xc vào [x1, x2], yc vào [y1, y2]
    pass

# Bài 4 — TODO
def circle_contains_rect(r, xc, yc, x1, y1, x2, y2):
    # TODO: góc xa nhất: x = x1 hoặc x2 (cái nào xa xc hơn), tương tự y
    pass

# Bài 5 — TODO
def count_common_lattice(r, xc, yc, x1, y1, x2, y2):
    # TODO: với mỗi x trong [max(x1, xc-r), min(x2, xc+r)]: h = isqrt(r*r - (x-xc)**2)
    # TODO: cộng số y nguyên trong [max(y1, yc-h), min(y2, yc+h)]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `Q = (1, −1)`, `d² = 4`. (2) `r = 2` ⇒ `r² = 4` ⇒ chạm ⇒ `True`. (3) `y2 = −2` ⇒ `Q = (1, −2)`, `d² = 9 > 4` ⇒ `False`.

**Bài 2.** (1) `Q = (clamp(7,0,4), clamp(−3,0,6)) = (4, 0)`. (2) `dx = 3`, `dy = 3`, `d² = 18`. (3) `18 ≤ 25` ⇒ `True`.

**Bài 3.** (1) `max(x1, min(xc, x2))`. (2) Tương tự cho `y`. (3) Trả tuple.

**Bài 4.** (1) `fx = max(abs(x1 − xc), abs(x2 − xc))`. (2) `fy` tương tự. (3) `fx² + fy² ≤ r²` (góc xa nhất trong tròn ⇒ cả hình chữ nhật trong tròn vì hình tròn lồi).

**Bài 5.** (1) Duyệt `x` trong giao theo trục x. (2) `h = isqrt(r² − (x − xc)²)` là nửa chiều cao nguyên của hình tròn ở cột `x`. (3) Cộng `max(0, hi − lo + 1)`; đối chiếu với đếm vét cạn.

## 19. Đáp án tham khảo

In [28]:
# Bài 1, 2
assert Solution().checkOverlap(2, 1, 1, 1, -3, 2, -1) is True
assert Solution().checkOverlap(2, 1, 1, 1, -3, 2, -2) is False
qx, qy = clamp(7, 0, 4), clamp(-3, 0, 6)
assert (qx, qy) == (4, 0) and (qx - 7) ** 2 + (qy + 3) ** 2 == 18
assert Solution().checkOverlap(5, 7, -3, 0, 0, 4, 6) is True
print("Bài 1, 2: đúng")

Bài 1, 2: đúng


In [29]:
# Bài 3 — đáp án
def closest_point_sol(xc, yc, x1, y1, x2, y2):
    return max(x1, min(xc, x2)), max(y1, min(yc, y2))

assert closest_point_sol(0, 0, 1, -1, 3, 1) == (1, 0)
assert closest_point_sol(1, 1, 1, -3, 2, -1) == (1, -1)
assert closest_point_sol(0, 0, -5, -5, 5, 5) == (0, 0)
rng = random.Random(3)
for _ in range(300):
    xc, yc = rng.randint(-6, 6), rng.randint(-6, 6)
    x1 = rng.randint(-6, 5); x2 = rng.randint(x1 + 1, 6); y1 = rng.randint(-6, 5); y2 = rng.randint(y1 + 1, 6)
    q = closest_point_sol(xc, yc, x1, y1, x2, y2)
    best = min((x - xc) ** 2 + (y - yc) ** 2 for x in range(x1, x2 + 1) for y in range(y1, y2 + 1))
    assert (q[0] - xc) ** 2 + (q[1] - yc) ** 2 == best
print("Bài 3: đúng")

Bài 3: đúng


In [30]:
# Bài 4 — đáp án
def circle_contains_rect_sol(r, xc, yc, x1, y1, x2, y2):
    fx = max(abs(x1 - xc), abs(x2 - xc))
    fy = max(abs(y1 - yc), abs(y2 - yc))
    return fx * fx + fy * fy <= r * r

def contains_brute(r, xc, yc, x1, y1, x2, y2):
    # đủ kiểm tra 4 góc vì hình tròn lồi; vét cạn mọi điểm nguyên để đối chiếu
    return all((x - xc) ** 2 + (y - yc) ** 2 <= r * r
               for x in range(x1, x2 + 1) for y in range(y1, y2 + 1))

rng = random.Random(4)
for _ in range(1000):
    r = rng.randint(1, 8); xc, yc = rng.randint(-5, 5), rng.randint(-5, 5)
    x1 = rng.randint(-6, 5); x2 = rng.randint(x1 + 1, 6); y1 = rng.randint(-6, 5); y2 = rng.randint(y1 + 1, 6)
    a = (r, xc, yc, x1, y1, x2, y2)
    assert circle_contains_rect_sol(*a) == contains_brute(*a)
print(circle_contains_rect_sol(10, 0, 0, -1, -1, 1, 1), circle_contains_rect_sol(1, 0, 0, -1, -1, 1, 1))
print("Bài 4: đúng")

True False
Bài 4: đúng


In [31]:
# Bài 5 — đáp án
def count_common_lattice_sol(r, xc, yc, x1, y1, x2, y2):
    total = 0
    for x in range(max(x1, xc - r), min(x2, xc + r) + 1):
        h = math.isqrt(r * r - (x - xc) ** 2)
        lo, hi = max(y1, yc - h), min(y2, yc + h)
        total += max(0, hi - lo + 1)
    return total

def count_common_brute(r, xc, yc, x1, y1, x2, y2):
    return sum(1 for x in range(x1, x2 + 1) for y in range(y1, y2 + 1)
               if (x - xc) ** 2 + (y - yc) ** 2 <= r * r)

rng = random.Random(5)
for _ in range(1000):
    r = rng.randint(1, 9); xc, yc = rng.randint(-8, 8), rng.randint(-8, 8)
    x1 = rng.randint(-10, 9); x2 = rng.randint(x1 + 1, 10); y1 = rng.randint(-10, 9); y2 = rng.randint(y1 + 1, 10)
    a = (r, xc, yc, x1, y1, x2, y2)
    c = count_common_lattice_sol(*a)
    assert c == count_common_brute(*a)
    assert (c > 0) == Solution().checkOverlap(*a)          # liên hệ với bài gốc (dữ liệu nguyên)
print("điểm nguyên trong hình tròn r=2000 nằm trong [-10^4,10^4]^2:",
      count_common_lattice_sol(2000, 0, 0, -10**4, -10**4, 10**4, 10**4))
print("Bài 5: đúng")

điểm nguyên trong hình tròn r=2000 nằm trong [-10^4,10^4]^2: 12566345
Bài 5: đúng


Lưu ý dòng `assert (c > 0) == checkOverlap(...)`: đây chính là kiểm chứng thực nghiệm cho nhận xét ở mục 5 — với dữ liệu nguyên, hai hình giao nhau **khi và chỉ khi** có điểm nguyên chung.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
qx = max(x1, min(xc, x2))        # max của cận dưới, min của cận trên
qy = max(y1, min(yc, y2))
return (qx - xc)**2 + (qy - yc)**2 <= r*r     # so bình phương, <= vì chạm cũng tính
```

Dạng tương đương: `dx = max(x1 - xc, 0, xc - x2)`, `dy = max(y1 - yc, 0, yc - y2)`, `dx*dx + dy*dy <= r*r`.

**Checklist trước khi submit:**
- [ ] Dùng điểm gần nhất (clamp), không chỉ kiểm tra góc/tâm.
- [ ] Clamp đúng thứ tự `max(lo, min(v, hi))`.
- [ ] So **bình phương**, không dùng `sqrt`.
- [ ] Dùng `<=` (tiếp xúc tính là giao).
- [ ] Đã thử: tròn trong chữ nhật, chữ nhật trong tròn, cắt giữa cạnh, chạm góc.
- [ ] Đúng chữ ký: `checkOverlap(self, radius, xCenter, yCenter, x1, y1, x2, y2) -> bool`.